In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm = ChatOllama(
    model = "llama3.2:3b"
)
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, model='llama3.2:3b')

In [22]:
"""
KOMBINASI 4: DENGAN reducer, DENGAN memory
"""
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    hasil: Annotated[list, operator.add]   # DENGAN reducer

def node1(state: State):
    return {"hasil": ["dari node1"]}

def node2(state: State):
    return {"hasil": ["dari node2"]}

g = StateGraph(State)
g.add_node("node1", node1)
g.add_node("node2", node2)
g.add_edge(START, "node1")
g.add_edge("node1", "node2")
g.add_edge("node2", END)

memory = InMemorySaver()
graph = g.compile(checkpointer=memory)   # DENGAN checkpointer

config = {"configurable": {"thread_id": "percobaan-1"}}

result1 = graph.invoke({"hasil": []}, config=config)
print("Invoke 1:", result1["hasil"])

result2 = graph.invoke({"hasil": []}, config=config)   # thread_id SAMA
print("Invoke 2:", result2["hasil"])

print("State history:", list(graph.get_state_history(config)))   # thread_id SAMA

Invoke 1: ['dari node1', 'dari node2']
Invoke 2: ['dari node1', 'dari node2', 'dari node1', 'dari node2']
State history: [StateSnapshot(values={'hasil': ['dari node1', 'dari node2', 'dari node1', 'dari node2']}, next=(), config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f4-ff86-6d5c-8006-c7ce6f9860a1'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-08-23T20:14:57.915631+00:00', parent_config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f4-ff83-6422-8005-eca63f7b33fb'}}, tasks=(), interrupts=()), StateSnapshot(values={'hasil': ['dari node1', 'dari node2', 'dari node1']}, next=('node2',), config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f4-ff83-6422-8005-eca63f7b33fb'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, created_at='2026-08-23T20:14:57.914166+00:00', parent_config={'configurable': {'thread_id': 'perc

In [23]:
"""
KOMBINASI 3: TANPA reducer, DENGAN memory
"""
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    hasil: list   # TANPA reducer

def node1(state: State):
    return {"hasil": ["dari node1"]}

def node2(state: State):
    return {"hasil": ["dari node2"]}

g = StateGraph(State)
g.add_node("node1", node1)
g.add_node("node2", node2)
g.add_edge(START, "node1")
g.add_edge("node1", "node2")
g.add_edge("node2", END)

memory = InMemorySaver()
graph = g.compile(checkpointer=memory)   # DENGAN checkpointer

config = {"configurable": {"thread_id": "percobaan-1"}}

result1 = graph.invoke({"hasil": []}, config=config)
print("Invoke 1:", result1["hasil"])

result2 = graph.invoke({"hasil": []}, config=config)   # thread_id SAMA
print("Invoke 2:", result2["hasil"])

print("State history:", list(graph.get_state_history(config)))   # thread_id SAMA

Invoke 1: ['dari node2']
Invoke 2: ['dari node2']
State history: [StateSnapshot(values={'hasil': ['dari node2']}, next=(), config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f6-8cac-6bb3-8006-8e65749470f0'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-08-23T20:15:39.559621+00:00', parent_config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f6-8caa-64a4-8005-e64702291f08'}}, tasks=(), interrupts=()), StateSnapshot(values={'hasil': ['dari node1']}, next=('node2',), config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f6-8caa-64a4-8005-e64702291f08'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, created_at='2026-08-23T20:15:39.558621+00:00', parent_config={'configurable': {'thread_id': 'percobaan-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f2f6-8ca7-6d96-8004-168c6551d42a'}}, tasks=(PregelTask(id='d20da051-fa04-

In [18]:
"""
KOMBINASI 2: DENGAN reducer, TANPA memory
"""
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    hasil: Annotated[list, operator.add]   # DENGAN reducer

def node1(state: State):
    return {"hasil": ["dari node1"]}

def node2(state: State):
    return {"hasil": ["dari node2"]}

g = StateGraph(State)
g.add_node("node1", node1)
g.add_node("node2", node2)
g.add_edge(START, "node1")
g.add_edge("node1", "node2")
g.add_edge("node2", END)

graph = g.compile()   # TANPA checkpointer

result1 = graph.invoke({"hasil": []})
print("Invoke 1:", result1["hasil"])

result2 = graph.invoke({"hasil": []})   # TANPA config/thread_id, gak ada memory
print("Invoke 2:", result2["hasil"])

Invoke 1: ['dari node1', 'dari node2']
Invoke 2: ['dari node1', 'dari node2']


In [19]:
"""
KOMBINASI 1: TANPA reducer, TANPA memory
"""
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    hasil: list   # TANPA reducer

def node1(state: State):
    return {"hasil": ["dari node1"]}

def node2(state: State):
    return {"hasil": ["dari node2"]}

g = StateGraph(State)
g.add_node("node1", node1)
g.add_node("node2", node2)
g.add_edge(START, "node1")
g.add_edge("node1", "node2")
g.add_edge("node2", END)

graph = g.compile()   # TANPA checkpointer

result1 = graph.invoke({"hasil": []})
print("Invoke 1:", result1["hasil"])

result2 = graph.invoke({"hasil": []})   # TANPA config/thread_id, gak ada memory
print("Invoke 2:", result2["hasil"])

Invoke 1: ['dari node2']
Invoke 2: ['dari node2']
